# Membuat SparkSession

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("RESPONSI UTS") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 14:34:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


## LATIHAN MANDIRI

In [4]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


## Soal 1 Kategori dengan Qty Tertinggi

In [5]:
df_gabungan.createOrReplaceTempView("gabungan")

spark.sql('''
    SELECT kategori, SUM(qty) AS total_qty
    FROM gabungan
    GROUP BY kategori
    ORDER BY total_qty DESC
    LIMIT 1
''').show()

[Stage 12:>                                                         (0 + 1) / 1]

+----------+---------+
|  kategori|total_qty|
+----------+---------+
|Elektronik|     6736|
+----------+---------+



## Soal 2 Toko Terbaik per Kategori

In [6]:
ringkasan_kategori_toko = df_gabungan.groupBy("kategori", "nama_toko").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_kategori = Window.partitionBy("kategori").orderBy(col("total_penjualan").desc())

ringkasan_kategori_toko.withColumn("peringkat", row_number().over(window_kategori)) \
    .filter(col("peringkat") == 1) \
    .orderBy("kategori") \
    .show(truncate=False)

+----------+-------------+---------------+---------+
|kategori  |nama_toko    |total_penjualan|peringkat|
+----------+-------------+---------------+---------+
|Elektronik|Toko Magelang|202150000      |1        |
|Fashion   |Toko Semarang|123000000      |1        |
|Makanan   |Toko Magelang|108200000      |1        |
+----------+-------------+---------------+---------+



## Soal 3 Simpan Ringkasan Eksekutif sebagai Parquet

In [8]:
ringkasan_eksekutif = df_gabungan.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

!hdfs dfs -mkdir -p /user/mahasiswa/latihan7

ringkasan_eksekutif.write.mode("overwrite").parquet(
    "hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif"
)

!hdfs dfs -ls /user/mahasiswa/latihan7/ringkasan_eksekutif

Found 2 items
-rw-r--r--   3 xiuviu supergroup          0 2026-10-10 14:25 /user/mahasiswa/latihan7/ringkasan_eksekutif/_SUCCESS
-rw-r--r--   3 xiuviu supergroup       2087 2026-10-10 14:25 /user/mahasiswa/latihan7/ringkasan_eksekutif/part-00000-9d7b4870-6da2-44dc-87a9-a35063aa2a13-c000.snappy.parquet


## Soal 4 Refleksi
**Jelaskan dalam 4 sampai 5 kalimat bagaimana HDFS, Spark, dan Parquet saling melengkapi.**

Jawab :

`HDFS menyimpan data mentah berukuran besar di banyak node (distributed storage). HDFS memecah file menjadi blok dan menyalinnya ke beberapa node, jadi data tetap ada jika satu node mati. Spark membaca data itu dan mengolahnya di memori (in-memory processing) dengan banyak proses paralel, sehingga join dan window function berjalan lebih cepat dibanding MapReduce yang menulis hasil antara ke disk. Spark lalu menyimpan hasilnya kembali ke HDFS sebagai Parquet berformat kolom (columnar format), sehingga file lebih kecil dan kueri hanya membaca kolom yang dibutuhkan. Dengan partitioning per kategori, Spark membuka folder yang relevan saja (partition pruning).`

___
# TUGAS MANDIRI

## Menyiapkan Dataset

In [7]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


## A. Extract

In [8]:
df_trx = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_trx.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


## B. Transform

In [9]:
df_gabungan = df_trx.join(df_cabang, on="cabang_id", how="inner") \
                    .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

print("Jumlah baris setelah join:", df_gabungan.count())
df_gabungan.show(5)

Jumlah baris setelah join: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makana

## C. Transform

In [10]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show(truncate=False)

+-------------------------+---------+--------------+---------------+---------+
|nama_cabang              |nama_menu|kategori_menu |total_penjualan|peringkat|
+-------------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|8505000        |1        |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|8085000        |2        |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|8750000        |1        |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|8715000        |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|9275000        |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |8160000        |2        |
+-------------------------+---------+--------------+---------------+---------+



## D. Transform

In [11]:
df_gabungan.createOrReplaceTempView("gabungan_kopi")

spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''').show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



## E. Load

In [12]:
!hdfs dfs -mkdir -p /user/xiuviu/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/xiuviu/tugas7/processed/laporan"
)

!hdfs dfs -ls -R /user/xiuviu/tugas7/processed/laporan

-rw-r--r--   3 xiuviu supergroup          0 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - xiuviu supergroup          0 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Kopi
-rw-r--r--   3 xiuviu supergroup      12309 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Kopi/part-00000-15f8a50b-9083-48c4-9fde-213a5b0c9ef5.c000.snappy.parquet
drwxr-xr-x   - xiuviu supergroup          0 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Makanan Ringan
-rw-r--r--   3 xiuviu supergroup      16393 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Makanan Ringan/part-00000-15f8a50b-9083-48c4-9fde-213a5b0c9ef5.c000.snappy.parquet
drwxr-xr-x   - xiuviu supergroup          0 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Non-Kopi
-rw-r--r--   3 xiuviu supergroup       8552 2026-10-10 14:39 /user/xiuviu/tugas7/processed/laporan/kategori_menu=Non-Kopi/part-00000-15f8a50b-

In [14]:
# Melihat isi
df_final = spark.read.parquet("hdfs://localhost:9000/user/xiuviu/tugas7/processed/laporan")

print("Jumlah baris hasil akhir      :", df_final.count())
print("Jumlah baris sebelum disimpan :", df_gabungan.count())

Jumlah baris hasil akhir      : 4000
Jumlah baris sebelum disimpan : 4000


## F. Ringkasan Eksekutif

In [15]:
ringkasan_cabang = df_final.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_cabang.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



## Menutup SparkSession

In [16]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
